# IAM Handwritten Text Training (Kaggle Ready)

This notebook contains complete code from data loading to final model save.

In [ ]:
# Optional install in Kaggle
# !pip install -q torch torchvision pillow

## 1) Imports, Seed, Device

In [ ]:
import os
import random
import shutil
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import transforms

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU is required for this notebook. In Kaggle, enable GPU accelerator and re-run.')

torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
DEVICE = torch.device('cuda')
print('Device:', DEVICE)
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))

## 2) Resolve Dataset Paths (Local or Kaggle)

In [ ]:
def first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None

def find_file_by_name(search_roots, filename):
    for root in search_roots:
        root = Path(root)
        if not root.exists():
            continue
        matches = list(root.rglob(filename))
        if matches:
            return matches[0]
    return None

def find_dir_by_name(search_roots, dirname):
    for root in search_roots:
        root = Path(root)
        if not root.exists():
            continue
        for p in root.rglob(dirname):
            if p.is_dir():
                return p
    return None

search_roots = [Path.cwd(), Path('/kaggle/input'), Path('/kaggle/working')]

IMAGE_DIR = first_existing([
    Path('data/iam_dataset/words'),
    Path('/kaggle/working/data/iam_dataset/words'),
]) or find_dir_by_name(search_roots, 'words')

TRAIN_LABEL_FILE = first_existing([
    Path('data/iam_dataset/train_gt.txt'),
    Path('/kaggle/working/data/iam_dataset/train_gt.txt'),
]) or find_file_by_name(search_roots, 'train_gt.txt')

VAL_LABEL_FILE = first_existing([
    Path('data/iam_dataset/val_gt.txt'),
    Path('/kaggle/working/data/iam_dataset/val_gt.txt'),
]) or find_file_by_name(search_roots, 'val_gt.txt')

OUTPUT_DIR = Path('/kaggle/working/model') if Path('/kaggle/working').exists() else Path('model')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / 'model.pth'
BEST_OUTPUT_PATH = OUTPUT_DIR / 'model_best.pth'

print('IMAGE_DIR      :', IMAGE_DIR)
print('TRAIN_LABEL_FILE:', TRAIN_LABEL_FILE)
print('VAL_LABEL_FILE :', VAL_LABEL_FILE)
print('OUTPUT_PATH    :', OUTPUT_PATH)

if IMAGE_DIR is None or TRAIN_LABEL_FILE is None:
    raise FileNotFoundError('Could not locate IAM image directory and/or train_gt.txt.')

## 3) Character Set + Dataset + Dataloader

In [ ]:
CHAR_SET = "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789 .,;:'\"!?()-/&%$#@+*=<>[]{}"
char_to_idx = {c: i + 1 for i, c in enumerate(CHAR_SET)}
idx_to_char = {v: k for k, v in char_to_idx.items()}

class IAMDataset(Dataset):
    def __init__(self, root_dir, label_file):
        self.samples = []
        root_dir = str(root_dir)
        label_file = str(label_file)

        if not os.path.isdir(root_dir):
            raise FileNotFoundError(f'Image directory not found: {root_dir}')
        if not os.path.isfile(label_file):
            raise FileNotFoundError(f'Label file not found: {label_file}')

        with open(label_file, 'r', encoding='utf-8', errors='ignore') as f:
            for line in f:
                if line.startswith('#'):
                    continue
                parts = line.strip().split()
                if not parts:
                    continue

                if len(parts) >= 9 and parts[1] in {'ok', 'err'}:
                    if parts[1] != 'ok':
                        continue
                    img_id = parts[0]
                    label = ' '.join(parts[8:]).replace('|', ' ')
                    img_path = os.path.join(root_dir, img_id + '.png')
                elif len(parts) >= 2:
                    rel_img_path = parts[0].replace('/', os.sep).replace('\\', os.sep)
                    label = line.strip()[len(parts[0]):].strip()
                    img_path = os.path.join(os.path.dirname(label_file), rel_img_path)
                else:
                    continue

                label = ''.join(c for c in label if c in char_to_idx)
                if not label.strip():
                    continue

                if os.path.exists(img_path):
                    self.samples.append((img_path, label))

        if not self.samples:
            raise RuntimeError('No usable samples found from labels.')

        self.transform = transforms.Compose([
            transforms.Grayscale(),
            transforms.Resize((32, 128)),
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,)),
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('L')
        return self.transform(image), label

def encode_text(text):
    return [char_to_idx[c] for c in text if c in char_to_idx]

def iam_collate_fn(batch):
    images, texts = zip(*batch)
    images = torch.stack(images)
    encoded = [torch.tensor(encode_text(t), dtype=torch.long) for t in texts]
    labels = torch.cat(encoded)
    label_lengths = torch.tensor([len(e) for e in encoded], dtype=torch.long)
    return images, labels, label_lengths, list(texts)

def create_iam_dataloader(dataset, batch_size=8, shuffle=True, num_workers=2):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        collate_fn=iam_collate_fn,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )

## 4) CRNN Model + Weight Initialization

In [ ]:
def init_weights(module):
    if isinstance(module, nn.Conv2d):
        nn.init.kaiming_normal_(module.weight, mode='fan_out', nonlinearity='relu')
        if module.bias is not None:
            nn.init.constant_(module.bias, 0.0)
    elif isinstance(module, nn.Linear):
        nn.init.xavier_uniform_(module.weight)
        if module.bias is not None:
            nn.init.constant_(module.bias, 0.0)
    elif isinstance(module, nn.LSTM):
        for name, param in module.named_parameters():
            if 'weight_ih' in name:
                nn.init.xavier_uniform_(param.data)
            elif 'weight_hh' in name:
                nn.init.orthogonal_(param.data)
            elif 'bias' in name:
                nn.init.constant_(param.data, 0.0)

class CRNN(nn.Module):
    def __init__(self, num_classes, dropout=0.2):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.GELU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(dropout),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.GELU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(dropout),
        )
        self.rnn = nn.LSTM(
            input_size=128 * 8,
            hidden_size=256,
            num_layers=2,
            bidirectional=True,
            dropout=dropout,
            batch_first=True,
        )
        self.head_dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(512, num_classes)
        self.apply(init_weights)

    def forward(self, x):
        x = self.cnn(x)
        b, c, h, w = x.size()
        x = x.permute(0, 3, 1, 2)
        x = x.reshape(b, w, c * h)
        x, _ = self.rnn(x)
        x = self.head_dropout(x)
        x = self.fc(x)
        return x.log_softmax(2)

model = CRNN(num_classes=len(char_to_idx) + 1, dropout=0.2).to(DEVICE)
print('Model initialized on', DEVICE)

## 5) Metrics + Evaluation

In [ ]:
def ctc_greedy_decode(log_probs_tnc):
    best = log_probs_tnc.argmax(2).transpose(0, 1)
    texts = []
    for seq in best:
        prev = -1
        out = []
        for token in seq.tolist():
            if token != prev and token != 0:
                out.append(idx_to_char.get(token, ''))
            prev = token
        texts.append(''.join(out))
    return texts

def _edit_distance(seq1, seq2):
    m, n = len(seq1), len(seq2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if seq1[i - 1] == seq2[j - 1] else 1
            dp[i][j] = min(dp[i - 1][j] + 1, dp[i][j - 1] + 1, dp[i - 1][j - 1] + cost)
    return dp[m][n]

def character_accuracy(pred_texts, true_texts):
    matched, total = 0, 0
    for pred, true in zip(pred_texts, true_texts):
        total += max(len(pred), len(true))
        matched += sum(1 for p, t in zip(pred, true) if p == t)
    return 0.0 if total == 0 else matched / total

def word_accuracy(pred_texts, true_texts):
    if not true_texts:
        return 0.0
    return sum(1 for p, t in zip(pred_texts, true_texts) if p == t) / len(true_texts)

def character_error_rate(pred_texts, true_texts):
    total_edits, total_chars = 0, 0
    for pred, true in zip(pred_texts, true_texts):
        total_edits += _edit_distance(pred, true)
        total_chars += max(len(true), 1)
    return 0.0 if total_chars == 0 else total_edits / total_chars

def word_error_rate(pred_texts, true_texts):
    total_edits, total_words = 0, 0
    for pred, true in zip(pred_texts, true_texts):
        total_edits += _edit_distance(pred.split(), true.split())
        total_words += max(len(true.split()), 1)
    return 0.0 if total_words == 0 else total_edits / total_words

def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    total_char_acc = 0.0
    total_word_acc = 0.0
    total_cer = 0.0
    total_wer = 0.0
    batches = 0
    with torch.no_grad():
        for images, labels, label_lengths, texts in loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            label_lengths = label_lengths.to(DEVICE, non_blocking=True)
            log_probs = model(images)
            time_steps = log_probs.size(1)
            input_lengths = torch.full((images.size(0),), time_steps, dtype=torch.long, device=DEVICE)
            preds = log_probs.permute(1, 0, 2)
            loss = criterion(preds, labels, input_lengths, label_lengths)
            pred_texts = ctc_greedy_decode(preds)
            total_loss += loss.item()
            total_char_acc += character_accuracy(pred_texts, texts)
            total_word_acc += word_accuracy(pred_texts, texts)
            total_cer += character_error_rate(pred_texts, texts)
            total_wer += word_error_rate(pred_texts, texts)
            batches += 1
    return {
        'loss': total_loss / max(batches, 1),
        'char_acc': total_char_acc / max(batches, 1),
        'word_acc': total_word_acc / max(batches, 1),
        'cer': total_cer / max(batches, 1),
        'wer': total_wer / max(batches, 1),
    }

## 6) Training Config

In [ ]:
EPOCHS = 20
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
STEP_SIZE = 5
LR_GAMMA = 0.5
EARLY_STOPPING_PATIENCE = 6
EARLY_STOPPING_MIN_DELTA = 1e-4
MAX_SAMPLES = 0
NUM_WORKERS = 2
USE_AMP = True
RESUME_TRAINING = True
FULL_CHECKPOINT_PATH = OUTPUT_DIR / 'last_checkpoint.pt'

print('EPOCHS:', EPOCHS)
print('BATCH_SIZE:', BATCH_SIZE)
print('LEARNING_RATE:', LEARNING_RATE)
print('WEIGHT_DECAY:', WEIGHT_DECAY)
print('STEP_SIZE:', STEP_SIZE)
print('LR_GAMMA:', LR_GAMMA)
print('EARLY_STOPPING_PATIENCE:', EARLY_STOPPING_PATIENCE)
print('USE_AMP:', USE_AMP)
print('RESUME_TRAINING:', RESUME_TRAINING)
print('FULL_CHECKPOINT_PATH:', FULL_CHECKPOINT_PATH)

## 7) Build Datasets and Dataloaders

In [ ]:
train_dataset = IAMDataset(str(IMAGE_DIR), str(TRAIN_LABEL_FILE))
print('Total train samples:', len(train_dataset))

if MAX_SAMPLES > 0 and MAX_SAMPLES < len(train_dataset):
    indices = list(range(len(train_dataset)))
    random.Random(SEED).shuffle(indices)
    train_dataset = Subset(train_dataset, indices[:MAX_SAMPLES])
    print('Using subset:', len(train_dataset))

train_loader = create_iam_dataloader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)

val_loader = None
if VAL_LABEL_FILE is not None and Path(VAL_LABEL_FILE).exists():
    val_dataset = IAMDataset(str(IMAGE_DIR), str(VAL_LABEL_FILE))
    val_loader = create_iam_dataloader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    print('Total val samples:', len(val_dataset))
else:
    print('Validation file not found. Running without validation.')

## 8) Resume Checkpoint Info

In [ ]:
if FULL_CHECKPOINT_PATH.exists():
    print('Resume checkpoint found:', FULL_CHECKPOINT_PATH)
else:
    print('No resume checkpoint found. Fresh training will start.')

## 9) Train and Save

In [ ]:
criterion = nn.CTCLoss(blank=0, zero_infinity=True).to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=STEP_SIZE, gamma=LR_GAMMA)
scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)

history = []
best_score = float('inf')
epochs_without_improvement = 0
start_epoch = 0

if RESUME_TRAINING and FULL_CHECKPOINT_PATH.exists():
    checkpoint = torch.load(str(FULL_CHECKPOINT_PATH), map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    if checkpoint.get('scaler_state_dict') is not None:
        scaler.load_state_dict(checkpoint['scaler_state_dict'])
    start_epoch = checkpoint.get('epoch', -1) + 1
    best_score = checkpoint.get('best_score', best_score)
    epochs_without_improvement = checkpoint.get('epochs_without_improvement', 0)
    history = checkpoint.get('history', [])
    print(f'Resumed from epoch {start_epoch}, best CER: {best_score:.4f}')

for epoch in range(start_epoch, EPOCHS):
    model.train()
    train_loss = 0.0
    train_char_acc = 0.0
    train_word_acc = 0.0
    train_cer = 0.0
    train_wer = 0.0
    batches = 0

    for images, labels, label_lengths, texts in train_loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        label_lengths = label_lengths.to(DEVICE, non_blocking=True)

        with torch.amp.autocast('cuda', enabled=USE_AMP):
            log_probs = model(images)
            time_steps = log_probs.size(1)
            input_lengths = torch.full((images.size(0),), time_steps, dtype=torch.long, device=DEVICE)
            preds = log_probs.permute(1, 0, 2)
            loss = criterion(preds, labels, input_lengths, label_lengths)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        scaler.step(optimizer)
        scaler.update()

        pred_texts = ctc_greedy_decode(preds.detach())
        train_loss += loss.item()
        train_char_acc += character_accuracy(pred_texts, texts)
        train_word_acc += word_accuracy(pred_texts, texts)
        train_cer += character_error_rate(pred_texts, texts)
        train_wer += word_error_rate(pred_texts, texts)
        batches += 1

    train_metrics = {
        'loss': train_loss / max(batches, 1),
        'char_acc': train_char_acc / max(batches, 1),
        'word_acc': train_word_acc / max(batches, 1),
        'cer': train_cer / max(batches, 1),
        'wer': train_wer / max(batches, 1),
    }

    current_lr = optimizer.param_groups[0]['lr']
    if val_loader is not None:
        val_metrics = evaluate(model, val_loader, criterion)
        score = val_metrics['cer']
        print(
            f"Epoch {epoch + 1}/{EPOCHS} | LR: {current_lr:.6f} | "
            f"Train Loss: {train_metrics['loss']:.4f}, Char Acc: {train_metrics['char_acc']:.4f}, "
            f"Word Acc: {train_metrics['word_acc']:.4f}, CER: {train_metrics['cer']:.4f}, WER: {train_metrics['wer']:.4f} | "
            f"Val Loss: {val_metrics['loss']:.4f}, Char Acc: {val_metrics['char_acc']:.4f}, "
            f"Word Acc: {val_metrics['word_acc']:.4f}, CER: {val_metrics['cer']:.4f}, WER: {val_metrics['wer']:.4f}"
        )
        epoch_log = {
            'epoch': epoch + 1,
            **{f'train_{k}': v for k, v in train_metrics.items()},
            **{f'val_{k}': v for k, v in val_metrics.items()},
        }
    else:
        score = train_metrics['cer']
        print(
            f"Epoch {epoch + 1}/{EPOCHS} | LR: {current_lr:.6f} | "
            f"Train Loss: {train_metrics['loss']:.4f}, Char Acc: {train_metrics['char_acc']:.4f}, "
            f"Word Acc: {train_metrics['word_acc']:.4f}, CER: {train_metrics['cer']:.4f}, WER: {train_metrics['wer']:.4f}"
        )
        epoch_log = {'epoch': epoch + 1, **{f'train_{k}': v for k, v in train_metrics.items()}}

    history.append(epoch_log)

    if score < (best_score - EARLY_STOPPING_MIN_DELTA):
        best_score = score
        epochs_without_improvement = 0
        torch.save(model.state_dict(), str(BEST_OUTPUT_PATH))
        print(f'Saved improved checkpoint: {BEST_OUTPUT_PATH} (CER={best_score:.4f})')
    else:
        epochs_without_improvement += 1
        print(f'No improvement for {epochs_without_improvement} epoch(s).')

    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict() if USE_AMP else None,
        'best_score': best_score,
        'epochs_without_improvement': epochs_without_improvement,
        'history': history,
    }, str(FULL_CHECKPOINT_PATH))

    scheduler.step()

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print('Early stopping triggered.')
        break

if BEST_OUTPUT_PATH.exists():
    model.load_state_dict(torch.load(str(BEST_OUTPUT_PATH), map_location=DEVICE))
    print('Loaded best checkpoint before final save.')

torch.save(model.state_dict(), str(OUTPUT_PATH))
print('Saved final model:', OUTPUT_PATH)
print('Saved best model :', BEST_OUTPUT_PATH)
print('Saved resume ckpt:', FULL_CHECKPOINT_PATH)

## 10) Keep model.pth synced with best checkpoint

In [ ]:
if BEST_OUTPUT_PATH.exists():
    shutil.copyfile(BEST_OUTPUT_PATH, OUTPUT_PATH)
    print('Copied best checkpoint to final model path.')

print('Final model exists:', OUTPUT_PATH.exists(), OUTPUT_PATH)
print('Best model exists :', BEST_OUTPUT_PATH.exists(), BEST_OUTPUT_PATH)

local_model_dir = Path('model')
local_model_dir.mkdir(exist_ok=True)
local_model_path = local_model_dir / 'model.pth'
if OUTPUT_PATH.resolve() != local_model_path.resolve():
    shutil.copyfile(OUTPUT_PATH, local_model_path)
    print('Also copied to:', local_model_path.resolve())

In [ ]:
# 11) Extended Evaluation Metrics: Accuracy, Precision, Recall, F1, ROC-AUC, PR-AUC, CER, WER
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    jaccard_score,
    cohen_kappa_score,
    hamming_loss,
    zero_one_loss,
    confusion_matrix,
    classification_report,
    fbeta_score,
    precision_recall_curve,
    roc_curve,
    auc
)

eval_loader = val_loader if val_loader is not None else train_loader
if eval_loader is None:
    raise RuntimeError('No dataloader available for evaluation.')

ckpt_path = BEST_OUTPUT_PATH if BEST_OUTPUT_PATH.exists() else OUTPUT_PATH
if not ckpt_path.exists():
    raise FileNotFoundError(f'Checkpoint not found: {ckpt_path}')

model.load_state_dict(torch.load(str(ckpt_path), map_location=DEVICE))
model.eval()

# Sequence-level metrics
word_correct = 0
word_total = 0
sum_cer = 0.0
sum_wer = 0.0

# Char-classification metrics on aligned character positions
char_true_labels = []
char_pred_labels = []

# Binary correctness metrics with confidence for ROC-AUC/PR-AUC
binary_true = []  # 1 if predicted char matches true char at same position
binary_score = []  # confidence of predicted char at that position

def ctc_decode_with_confidence(log_probs_ntc):
    probs = log_probs_ntc.exp()
    best_prob, best_idx = probs.max(dim=2)  # (N, T)
    all_texts = []
    all_confs = []
    for idx_seq, conf_seq in zip(best_idx.tolist(), best_prob.tolist()):
        prev = -1
        out_chars = []
        out_confs = []
        for token, conf in zip(idx_seq, conf_seq):
            if token != prev and token != 0:
                out_chars.append(idx_to_char.get(token, ''))
                out_confs.append(float(conf))
            prev = token
        all_texts.append(''.join(out_chars))
        all_confs.append(out_confs)
    return all_texts, all_confs

with torch.no_grad():
    for images, labels, label_lengths, texts in eval_loader:
        images = images.to(DEVICE, non_blocking=True)
        log_probs_ntc = model(images)  # (N, T, C)
        pred_texts, pred_confs = ctc_decode_with_confidence(log_probs_ntc)

        for pred, true, confs in zip(pred_texts, texts, pred_confs):
            word_correct += int(pred == true)
            word_total += 1

            sum_cer += character_error_rate([pred], [true])
            sum_wer += word_error_rate([pred], [true])

            aligned_len = min(len(pred), len(true), len(confs))
            for i in range(aligned_len):
                p = pred[i]
                t = true[i]
                c = confs[i]
                char_true_labels.append(t)
                char_pred_labels.append(p)
                binary_true.append(1 if p == t else 0)
                binary_score.append(c)

# Main sequence metrics
word_accuracy = (word_correct / word_total) if word_total > 0 else 0.0
cer = (sum_cer / word_total) if word_total > 0 else 0.0
wer = (sum_wer / word_total) if word_total > 0 else 0.0

# Char-level multiclass metrics
if len(char_true_labels) == 0:
    raise RuntimeError('No aligned characters available for metric computation.')

char_accuracy = accuracy_score(char_true_labels, char_pred_labels)
char_precision_micro = precision_score(char_true_labels, char_pred_labels, average='micro', zero_division=0)
char_recall_micro = recall_score(char_true_labels, char_pred_labels, average='micro', zero_division=0)
char_f1_micro = f1_score(char_true_labels, char_pred_labels, average='micro', zero_division=0)
char_f2_micro = fbeta_score(char_true_labels, char_pred_labels, average='micro', beta=2, zero_division=0)

char_precision_macro = precision_score(char_true_labels, char_pred_labels, average='macro', zero_division=0)
char_recall_macro = recall_score(char_true_labels, char_pred_labels, average='macro', zero_division=0)
char_f1_macro = f1_score(char_true_labels, char_pred_labels, average='macro', zero_division=0)

balanced_acc = balanced_accuracy_score(char_true_labels, char_pred_labels)
mcc = matthews_corrcoef(char_true_labels, char_pred_labels)
kappa = cohen_kappa_score(char_true_labels, char_pred_labels)
jaccard_micro = jaccard_score(char_true_labels, char_pred_labels, average='micro', zero_division=0)
ham_loss = hamming_loss(char_true_labels, char_pred_labels)
zo_loss = zero_one_loss(char_true_labels, char_pred_labels)

# Confidence-based binary correctness metrics
if len(set(binary_true)) >= 2:
    roc_auc = roc_auc_score(binary_true, binary_score)
    pr_auc = average_precision_score(binary_true, binary_score)
    fpr, tpr, _ = roc_curve(binary_true, binary_score)
    precision_curve, recall_curve, _ = precision_recall_curve(binary_true, binary_score)
    roc_auc_via_curve = auc(fpr, tpr)
    pr_auc_via_curve = auc(recall_curve, precision_curve)
else:
    roc_auc = float('nan')
    pr_auc = float('nan')
    roc_auc_via_curve = float('nan')
    pr_auc_via_curve = float('nan')

print('Checkpoint:', ckpt_path)
print('Samples   :', word_total)
print('')
print('Word-level:')
print(f'  Accuracy : {word_accuracy:.4f}')
print(f'  CER      : {cer:.4f}')
print(f'  WER      : {wer:.4f}')
print('')
print('Char-level (micro):')
print(f'  Accuracy : {char_accuracy:.4f}')
print(f'  Precision: {char_precision_micro:.4f}')
print(f'  Recall   : {char_recall_micro:.4f}')
print(f'  F1-score : {char_f1_micro:.4f}')
print(f'  F2-score : {char_f2_micro:.4f}')
print(f'  Jaccard  : {jaccard_micro:.4f}')
print('')
print('Char-level (macro):')
print(f'  Precision: {char_precision_macro:.4f}')
print(f'  Recall   : {char_recall_macro:.4f}')
print(f'  F1-score : {char_f1_macro:.4f}')
print(f'  Balanced Accuracy: {balanced_acc:.4f}')
print(f'  MCC      : {mcc:.4f}')
print(f'  Kappa    : {kappa:.4f}')
print(f'  Hamming Loss : {ham_loss:.4f}')
print(f'  Zero-One Loss: {zo_loss:.4f}')
print('')
print('Confidence-based correctness:')
print(f'  ROC-AUC       : {roc_auc:.4f}')
print(f'  PR-AUC        : {pr_auc:.4f}')
print(f'  ROC-AUC(curve): {roc_auc_via_curve:.4f}')
print(f'  PR-AUC(curve) : {pr_auc_via_curve:.4f}')

print('\nClassification report (char-level):')
print(classification_report(char_true_labels, char_pred_labels, zero_division=0))

print('Confusion matrix shape:', confusion_matrix(char_true_labels, char_pred_labels).shape)